# 4. Full Deep Research Agent with Virtual Files

Поисковый агент на основе архитектуры [Deep Agents from Scratch](https://github.com/langchain-ai/deep-agents-from-scratch).

## Возможности
- **Поиск в интернете** — DuckDuckGo через LangChain
- **Виртуальные файлы** — in-memory файловая система через `StateBackend`
- **Экспорт** — выгрузка виртуальных файлов в реальную ФС после завершения сессии

## Cell 1: Imports, environment loading, and dependency verification

In [ ]:
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify required packages
REQUIRED_PACKAGES = [
    "langchain",
    "langgraph",
    "langchain_community",
    "deepagents",
]

missing = []
for pkg in REQUIRED_PACKAGES:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    raise ImportError(
        f"Missing required packages: {', '.join(missing)}. "
        f"Install with: pip install {' '.join(missing)}"
    )

print("All required packages are available.")

## Cell 2: Virtual filesystem layer (StateBackend wrapper)

In [ ]:
from deepagents import StateBackend


class VirtualFileSystem:
    """In-memory virtual filesystem backed by StateBackend.
    
    Provides read/write/ls operations that the agent can use
    through built-in tools auto-wired by StateBackend.
    """
    
    def __init__(self):
        self.backend = StateBackend()
        self._store: dict[str, str] = {}
    
    def write(self, path: str, content: str) -> str:
        """Write content to a virtual file."""
        self._store[path] = content
        return f"File '{path}' written successfully ({len(content)} bytes)."
    
    def read(self, path: str) -> str:
        """Read content from a virtual file."""
        if path not in self._store:
            raise FileNotFoundError(f"Virtual file '{path}' not found.")
        return self._store[path]
    
    def ls(self) -> str:
        """List all virtual files."""
        if not self._store:
            return "No virtual files created yet."
        lines = ["Virtual files:"]
        for path in sorted(self._store.keys()):
            size = len(self._store[path])
            lines.append(f"  {path} ({size} bytes)")
        return "\n".join(lines)
    
    def get_all_files(self) -> dict[str, str]:
        """Return all stored virtual files for export."""
        return dict(self._store)


# Create the virtual filesystem instance
vfs = VirtualFileSystem()
print("Virtual filesystem initialized.")

## Cell 3: Web search tool definition

In [ ]:
from langchain_core.tools import tool


@tool
def internet_search(query: str, max_results: int = 5) -> str:
    """Search the internet for current information.
    
    Args:
        query: The search query string.
        max_results: Maximum number of results to return (default 5).
    
    Returns:
        Formatted search results as a string.
    """
    try:
        from langchain_community.tools import DuckDuckGoSearchResults
    except ImportError:
        return "Error: langchain-community not installed. Install with: pip install langchain-community"
    
    search = DuckDuckGoSearchResults(
        name="internet_search",
        description="Search the internet for current information.",
    )
    
    try:
        results = search.invoke({"query": query, "max_results": max_results})
        return str(results)
    except Exception as e:
        return f"Search error: {e}"


# Create virtual filesystem tools for the agent
@tool
def write_virtual_file(path: str, content: str) -> str:
    """Write content to a virtual file in memory.
    
    Args:
        path: The file path (e.g., 'report.txt', 'data/results.json').
        content: The text content to write.
    
    Returns:
        Confirmation message.
    """
    return vfs.write(path, content)


@tool
def read_virtual_file(path: str) -> str:
    """Read content from a virtual file.
    
    Args:
        path: The file path to read.
    
    Returns:
        The file content.
    """
    return vfs.read(path)


@tool
def list_virtual_files() -> str:
    """List all virtual files currently in memory.
    
    Returns:
        A formatted list of virtual files with sizes.
    """
    return vfs.ls()


print("Search tool and virtual filesystem tools created.")

## Cell 4: System prompt configuration & agent graph compilation

In [ ]:
from deepagents import create_deep_agent


SYSTEM_PROMPT = """You are a deep research assistant. You can search the internet for information
and create, read, and manage virtual files.

When the user asks a question:
1. Use the internet_search tool to find relevant information.
2. Summarize findings and optionally save them to virtual files using write_virtual_file.
3. You can list, read, and edit virtual files as needed.
4. Be thorough and cite sources when possible.

Virtual files are stored in memory and will be exported to the real filesystem
when the session ends.

Always save important research findings to virtual files so they can be exported later."""


def get_llm_model() -> str:
    """Return the LLM model identifier."""
    return os.getenv("LLM_MODEL", "openai:gpt-4o")


def create_agent():
    """Create and return a compiled deep agent with search and filesystem tools."""
    tools = [
        internet_search,
        write_virtual_file,
        read_virtual_file,
        list_virtual_files,
    ]
    
    agent = create_deep_agent(
        model=get_llm_model(),
        tools=tools,
        backend=StateBackend(),
        system_prompt=SYSTEM_PROMPT,
    )
    return agent


# Create the agent
try:
    agent = create_agent()
    print("Agent created successfully.")
except Exception as e:
    print(f"Error creating agent: {e}")
    print("Make sure you have configured your LLM API key in .env")
    agent = None

## Cell 5: Execution trigger & export routine

In [ ]:
from langchain_core.messages import HumanMessage


def run_agent_query(agent, user_query: str, thread_id: str) -> dict:
    """Run the agent with a user query and return the final state.
    
    Args:
        agent: Compiled LangGraph agent.
        user_query: The user's question or command.
        thread_id: Session identifier for state persistence.
    
    Returns:
        The final agent state after processing the query.
    """
    config = {"configurable": {"thread_id": thread_id}}
    messages = [HumanMessage(content=user_query)]
    response = agent.invoke({"messages": messages}, config=config)
    return response


def export_virtual_files(vfs_instance: VirtualFileSystem, export_dir: Path | None = None) -> list[Path]:
    """Export virtual files to the real filesystem.
    
    Args:
        vfs_instance: The VirtualFileSystem instance containing files.
        export_dir: Target directory. Defaults to ./exported_files/.
    
    Returns:
        List of exported file paths.
    """
    if export_dir is None:
        export_dir = Path(os.getenv("EXPORT_DIR", "./exported_files"))
    
    export_dir.mkdir(parents=True, exist_ok=True)
    exported: list[Path] = []
    
    virtual_files = vfs_instance.get_all_files()
    
    for file_path, content in virtual_files.items():
        target = (export_dir / file_path).resolve()
        base_resolved = export_dir.resolve()
        
        # Safety check: prevent directory traversal
        try:
            target.relative_to(base_resolved)
        except ValueError:
            print(f"  Blocked: '{file_path}' would escape export directory.")
            continue
        
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        exported.append(target)
    
    return exported


# --- Interactive session ---
thread_id = os.getenv("THREAD_ID", str(uuid.uuid4()))
print(f"Session thread_id: {thread_id}")
print("Type your query or '/exit' to finish.")

all_states: list[dict] = []

if agent is not None:
    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession interrupted.")
            break
        
        if not user_input:
            continue
        
        if user_input.lower() == "/exit":
            print("\nEnding session...")
            break
        
        try:
            state = run_agent_query(agent, user_input, thread_id)
            all_states.append(state)
            
            messages = state.get("messages", [])
            if messages:
                last_msg = messages[-1]
                content = getattr(last_msg, "content", str(last_msg))
                print(f"\nAgent: {content}\n")
        except Exception as e:
            print(f"\nError during agent execution: {e}\n")
    
    # Export virtual files
    print("\nExporting virtual files...")
    try:
        exported = export_virtual_files(vfs)
        if exported:
            print(f"Exported {len(exported)} file(s):")
            for f in exported:
                print(f"  - {f}")
        else:
            print("No virtual files to export.")
    except Exception as e:
        print(f"Error during export: {e}")
    
    print("\nSession ended.")
else:
    print("Agent not available. Configure your LLM API key and re-run.")